## 1. Configuration

In [1]:
from preprocessing.config import DownsampleConfig, EdgeConfig

# ---- Step 1: Downsampling ----
ds_cfg = DownsampleConfig(
    root="Dataset",              # AirfRANS 데이터셋 경로 (없으면 자동 다운로드)
    task="scarce",               # "scarce" (200 samples) / "full" (1000 samples)
    out_dir="downsampled_graphs",
    limit_train=None,            # 학습 그래프 수 제한 (None = 전체, 테스트 시 5 등으로 설정)
    limit_test=None,             # 테스트 그래프 수 제한
    target_min_nodes=15_000,     # 다운샘플링 목표 최소 노드 수
    target_max_nodes=30_000,     # 다운샘플링 목표 최대 노드 수
    voxel_frac=0.01,             # 초기 복셀 크기 = chord_length * voxel_frac
    voxel_iters=5,               # 적응적 복셀 크기 조정 반복 횟수
)

# ---- Step 2: Edge Construction ----
edge_cfg = EdgeConfig(
    in_dir="downsampled_graphs",  # Step 1 출력 경로
    out_dir="prebuilt_edges",     # 최종 출력 경로
    task="scarce",                # ds_cfg.task과 동일해야 함
    global_radius=0.02,           # 체적 노드 연결 반경
    surface_radius=0.01,          # 표면 노드 연결 반경 (더 촘촘)
    max_num_neighbors=48,         # 노드당 최대 이웃 수
    surface_ring=True,            # 표면 노드를 순서대로 연결 (에어포일 윤곽)
    denormalize=False,            # 정규화 역변환 여부
    min_degree=2,                 # 최소 차수 보장
    knn_backup_k=4,              # 고립 노드 KNN 백업 이웃 수
    knn_max_radius=0.05,         # KNN 백업 최대 반경
)

print("=" * 60)
print("Downsampling Config")
print("=" * 60)
for k, v in vars(ds_cfg).items():
    print(f"  {k:25s} = {v}")
print()
print("=" * 60)
print("Edge Construction Config")
print("=" * 60)
for k, v in vars(edge_cfg).items():
    print(f"  {k:25s} = {v}")

Downsampling Config
  root                      = Dataset
  task                      = scarce
  out_dir                   = downsampled_graphs
  limit_train               = None
  limit_test                = None
  target_min_nodes          = 15000
  target_max_nodes          = 30000
  voxel_frac                = 0.01
  voxel_iters               = 5

Edge Construction Config
  in_dir                    = downsampled_graphs
  out_dir                   = prebuilt_edges
  task                      = scarce
  global_radius             = 0.02
  surface_radius            = 0.01
  max_num_neighbors         = 48
  surface_ring              = True
  denormalize               = False
  min_degree                = 2
  knn_backup_k              = 4
  knn_max_radius            = 0.05


## 2. Step 1 — Adaptive Voxel Downsampling

원본 AirfRANS 메시(~100k 노드)를 15k~30k 노드로 다운샘플링합니다.

- 표면(airfoil) 노드는 **100% 보존**
- 체적 노드만 voxel grid로 간추림
- 복셀 크기를 적응적으로 조정하여 목표 노드 수 범위에 도달

In [2]:
from preprocessing.downsample_airfrans import run as run_downsample

run_downsample(ds_cfg)

/home/elicer/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Downsample -> downsampled_graphs/scarce/test: 100%|██████████| 200/200 [00:04<00:00, 42.70it/s]


Saved downsampled: train=200 test=200 under downsampled_graphs/scarce


### 2.1 Downsampling 결과 확인

In [3]:
import os
import torch

ds_out_root = os.path.join(ds_cfg.out_dir, ds_cfg.task)

for split in ("train", "test"):
    split_dir = os.path.join(ds_out_root, split)
    if not os.path.isdir(split_dir):
        print(f"[{split}] directory not found: {split_dir}")
        continue
    files = sorted(f for f in os.listdir(split_dir) if f.endswith(".pt"))
    print(f"[{split}] {len(files)} graphs saved")

    if len(files) > 0:
        node_counts = []
        for fn in files[:10]:  # 처음 10개만 확인
            d = torch.load(os.path.join(split_dir, fn), map_location="cpu", weights_only=False)
            n = d.x.size(0) if d.x is not None else (d.pos.size(0) if d.pos is not None else 0)
            node_counts.append(n)
        print(f"  Node counts (first {len(node_counts)}): "
              f"min={min(node_counts)}, max={max(node_counts)}, mean={sum(node_counts)/len(node_counts):.0f}")

        # 첫 번째 그래프 상세 정보
        d0 = torch.load(os.path.join(split_dir, files[0]), map_location="cpu", weights_only=False)
        print(f"  Sample graph keys: {list(d0.to_dict().keys())}")
        if d0.x is not None:
            print(f"  x shape: {d0.x.shape}  (node features)")
        if d0.y is not None:
            print(f"  y shape: {d0.y.shape}  (targets)")
    print()

[train] 200 graphs saved
  Node counts (first 10): min=15020, max=17987, mean=16779
  Sample graph keys: ['x', 'y', 'pos', 'surf', 'orig_index']
  x shape: torch.Size([17987, 5])  (node features)
  y shape: torch.Size([17987, 4])  (targets)

[test] 200 graphs saved
  Node counts (first 10): min=15009, max=18045, mean=17082
  Sample graph keys: ['x', 'y', 'pos', 'surf', 'orig_index']
  x shape: torch.Size([18002, 5])  (node features)
  y shape: torch.Size([18002, 4])  (targets)



## 3. Step 2 — Radius-Graph Edge Construction

다운샘플된 그래프에 에지와 에지 피처를 추가합니다.

- **Global radius graph** (r=0.02): 모든 노드 간 연결
- **Surface radius graph** (r=0.01): 표면 노드 간 추가 연결
- **Surface ring edges**: 에어포일 윤곽선 형성
- **KNN backup**: 고립 노드 최소 차수 보장
- **Edge features 5D**: [거리, 방향x, 방향y, cos(법선i, 법선j), 표면쌍 여부]

In [4]:
from preprocessing.edges_from_downsampled import run as run_edges

run_edges(edge_cfg)

Edges -> prebuilt_edges/scarce/train: 100%|██████████| 200/200 [07:37<00:00,  2.29s/it]


Saved edges for train: 200 files in prebuilt_edges/scarce/train


Edges -> prebuilt_edges/scarce/test: 100%|██████████| 200/200 [07:48<00:00,  2.34s/it]

Saved edges for test: 200 files in prebuilt_edges/scarce/test


### 3.1 Edge Construction 결과 확인

In [5]:
edge_out_root = os.path.join(edge_cfg.out_dir, edge_cfg.task)

for split in ("train", "test"):
    split_dir = os.path.join(edge_out_root, split)
    if not os.path.isdir(split_dir):
        print(f"[{split}] directory not found: {split_dir}")
        continue
    files = sorted(f for f in os.listdir(split_dir) if f.endswith(".pt"))
    print(f"[{split}] {len(files)} graphs with edges")

    if len(files) > 0:
        d0 = torch.load(os.path.join(split_dir, files[0]), map_location="cpu", weights_only=False)
        n_nodes = d0.x.size(0) if d0.x is not None else 0
        n_edges = d0.edge_index.size(1) if d0.edge_index is not None else 0
        avg_degree = n_edges / max(n_nodes, 1)

        print(f"  Sample graph:")
        print(f"    Nodes: {n_nodes}")
        print(f"    Edges: {n_edges}")
        print(f"    Avg degree: {avg_degree:.1f}")
        if d0.edge_attr is not None:
            print(f"    Edge features: {d0.edge_attr.shape}  (5D: dist, dir_x, dir_y, cos_normal, is_surface)")
        print(f"    Keys: {list(d0.to_dict().keys())}")
    print()

[train] 200 graphs with edges
  Sample graph:
    Nodes: 17987
    Edges: 68640
    Avg degree: 3.8
    Edge features: torch.Size([68640, 5])  (5D: dist, dir_x, dir_y, cos_normal, is_surface)
    Keys: ['x', 'y', 'pos', 'surf', 'orig_index', 'edge_index', 'edge_attr', 'edge_attr_dxdy', 'edge_meta']

[test] 200 graphs with edges
  Sample graph:
    Nodes: 18002
    Edges: 68008
    Avg degree: 3.8
    Edge features: torch.Size([68008, 5])  (5D: dist, dir_x, dir_y, cos_normal, is_surface)
    Keys: ['x', 'y', 'pos', 'surf', 'orig_index', 'edge_index', 'edge_attr', 'edge_attr_dxdy', 'edge_meta']



## 4. 완료

전처리가 완료되었습니다. 다음 단계로 학습을 시작할 수 있습니다:

```bash
python scripts/train.py --wandb-mode disabled --epochs 5  # 빠른 테스트
python scripts/train.py                                    # 기본 학습
```